# 7. 배치운영 및 재학습
 IF 점수로 우선 확인 순서를 정하고 LR·OCSVM·RF의 저장 모델로 배치 전체를 독립 추론합니다. IF에서 일부만 통과시키는 결합 필터는 구현하지 않으며 제품을 자동 폐기하지 않습니다.
 실제 제품 ID와 원본 연결, 패턴별 불량 이력, 모델 버전 및 입력 해시를 저장합니다. 라벨이 없는 배치는 추론·분포 감시만 수행하고 학습에는 사용하지 않습니다.
 재학습은 분포 변화 검토와 원인·실제 라벨 확인 후 선택 모델의 설정 및 임계값을 유지하여 전체 재적합합니다. 후보를 독립 평가한 뒤 명시적으로 전환하며 이전 모델로 복구할 수 있습니다.
 기본 실행은 상태 조회만 수행합니다. 이 노트북은 기존 runtime 폴더와 별개인 `output/batch_operations_runtime`에 상태를 저장합니다. 로컬 단일 실행용이며 기존 다중 실행·승인 운영 시스템을 대체하지 않습니다.

## 7.1 공통 환경·입력·지표

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, uuid, os, copy, warnings, platform
import numpy as np
import pandas as pd
import joblib, sklearn
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import average_precision_score, roc_auc_score
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path(r'C:\workspace\26KAMP')]
             if (p/'data/schema/input_features.json').is_file()), None)
if ROOT is None: raise FileNotFoundError('26KAMP 저장소 위치를 확인하세요.')
FEATURES = json.loads((ROOT/'data/schema/input_features.json').read_text(encoding='utf-8'))['feature_columns']
def read_json(path): return json.loads(Path(path).read_text(encoding='utf-8'))
def digest(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def timestamp(): return datetime.now(timezone.utc).isoformat()
def run_id(prefix): return prefix+'-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')+'-'+uuid.uuid4().hex[:8]
def write_json(path, value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    def convert(x):
        if isinstance(x,np.generic):return x.item()
        if isinstance(x,Path):return str(x)
        raise TypeError(type(x).__name__)
    tmp.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False,default=convert),encoding='utf-8')
    os.replace(tmp,path)
def validate_inputs(frame):
    if frame.empty or not set(FEATURES)<=set(frame): raise ValueError('입력 24개 변수 누락 또는 빈 자료')
    x=frame.loc[:,FEATURES].copy()
    if not all(pd.api.types.is_numeric_dtype(d) for d in x.dtypes):raise ValueError('입력은 숫자형이어야 합니다.')
    if not np.isfinite(x.to_numpy(dtype=float)).all():raise ValueError('결측 또는 무한값이 있습니다.')
    return x.astype(float)
def labels(values):
    y=pd.to_numeric(pd.Series(values),errors='raise').to_numpy()
    if not np.isfinite(y).all() or not set(y)<={0,1}:raise ValueError('라벨은 결측 없는 0/1이어야 합니다.')
    return y.astype(int)
def fingerprints(x):
    a=validate_inputs(x).to_numpy(dtype='<f8',copy=True);a[a==0]=0
    return [hashlib.sha256(row.tobytes()).hexdigest() for row in a]
def load_dataset(dataset):
    if dataset not in {'cn7','rg3'}:raise ValueError(dataset)
    folder=ROOT/f'data/processed/{dataset}/conservative'
    manifest=read_json(folder/'splits/split_manifest.json')
    for name,expected in manifest['source_sha256'].items():
        if digest(folder/name)!=expected:raise ValueError('분할 원본 해시 불일치: '+name)
    for name,expected in manifest['artifact_sha256'].items():
        if digest(folder/'splits'/name)!=expected:raise ValueError('분할 산출물 해시 불일치: '+name)
    x=pd.read_csv(folder/'X_labeled.csv',float_precision='round_trip')
    if list(x)!=FEATURES:raise ValueError('전처리 입력 순서가 스키마와 다릅니다.')
    x=validate_inputs(x);y=labels(pd.read_csv(folder/'y_labeled.csv').PassOrFail)
    split=pd.read_csv(folder/'splits/split_assignments.csv')
    if len(x)!=len(y) or not np.array_equal(split.pattern_row,np.arange(len(x))) or not np.array_equal(split.label,y):raise ValueError('분할/라벨 정렬 불일치')
    dev=split.loc[split.partition.eq('development'),'pattern_row'].to_numpy()
    test=split.loc[split.partition.eq('test'),'pattern_row'].to_numpy()
    if set(fingerprints(x.loc[dev]))&set(fingerprints(x.loc[test])):raise ValueError('개발/평가 입력 패턴 중복')
    return x,y,split,dev,test
def classification_metrics(y,p):
    y=labels(y);p=labels(p)
    if len(y)!=len(p):raise ValueError('정답과 예측 길이 불일치')
    tp=int(((y==1)&(p==1)).sum());fp=int(((y==0)&(p==1)).sum())
    fn=int(((y==1)&(p==0)).sum());tn=int(((y==0)&(p==0)).sum())
    return dict(TP=tp,FP=fp,FN=fn,TN=tn,F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.,
                recall=tp/(tp+fn) if tp+fn else None,precision=tp/(tp+fp) if tp+fp else None,
                FPR=fp/(fp+tn) if fp+tn else None)
def score_metrics(y,s):
    y=labels(y);s=np.asarray(s,dtype=float)
    return dict(AP=float(average_precision_score(y,s)) if y.sum() else None,
                ROC_AUC=float(roc_auc_score(y,s)) if len(set(y))==2 else None)

## 7.2 학습 단계의 변환 함수

In [ ]:
# 학습/탐색 셀을 실행하지 않고 각 모델의 변환을 별도 공간에 보존합니다.
rf_library = dict(globals())
exec(compile(r'''
SEED = 42
SCENARIOS = ('raw_no_scaler', 'without_cycle', 'pressure', 'plasticizing', 'thermal', 'domain_all', 'group_pca')
GROUPS = {'충전과 전환': ['Injection_Time', 'Filling_Time', 'Cushion_Position', 'Max_Injection_Speed', 'Max_Injection_Pressure', 'Max_Switch_Over_Pressure'], '계량과 가소화': ['Plasticizing_Time', 'Plasticizing_Position', 'Max_Screw_RPM', 'Average_Screw_RPM', 'Max_Back_Pressure', 'Average_Back_Pressure'], '실린더와 호퍼 온도': [f'Barrel_Temperature_{i}' for i in range(1, 7)] + ['Hopper_Temperature'], '금형 온도': ['Mold_Temperature_3', 'Mold_Temperature_4'], '형체와 전체 주기': ['Clamp_Close_Time', 'Clamp_Open_Position', 'Cycle_Time']}
DOMAIN_SPEC = [dict(feature='사출_전환압력_제공값차', group='pressure', operation='difference', columns=['Max_Injection_Pressure', 'Max_Switch_Over_Pressure']), dict(feature='스크루RPM_제공값차', group='plasticizing', operation='difference', columns=['Max_Screw_RPM', 'Average_Screw_RPM']), dict(feature='배압_제공값차', group='plasticizing', operation='difference', columns=['Max_Back_Pressure', 'Average_Back_Pressure']), dict(feature='배럴_제공값평균', group='thermal', operation='mean', columns=[f'Barrel_Temperature_{i}' for i in range(1, 7)]), dict(feature='배럴_제공값센서간산포', group='thermal', operation='std', columns=[f'Barrel_Temperature_{i}' for i in range(1, 7)]), dict(feature='금형_제공값평균', group='thermal', operation='mean', columns=['Mold_Temperature_3', 'Mold_Temperature_4']), dict(feature='금형온도_제공값차', group='thermal', operation='difference', columns=['Mold_Temperature_3', 'Mold_Temperature_4'])]
SCENARIO_GROUPS = {'raw_no_scaler': (), 'without_cycle': (), 'pressure': ('pressure',), 'plasticizing': ('plasticizing',), 'thermal': ('thermal',), 'domain_all': ('pressure', 'plasticizing', 'thermal'), 'group_pca': ()}

def derived_frame(coordinates: pd.DataFrame, specs: list[dict]) -> pd.DataFrame:
    result = pd.DataFrame(index=coordinates.index)
    for spec in specs:
        values = coordinates[spec['active_columns']]
        if spec['operation'] == 'difference':
            result[spec['feature']] = values.iloc[:, 0] - values.iloc[:, 1]
        elif spec['operation'] == 'mean':
            result[spec['feature']] = values.mean(axis=1)
        elif spec['operation'] == 'std':
            result[spec['feature']] = values.std(axis=1, ddof=0)
        else:
            raise ValueError(spec['operation'])
    return result

def feature_frame(bundle: dict, inputs: pd.DataFrame) -> pd.DataFrame:
    inputs = inputs.loc[:, bundle['input_columns']]
    scenario = bundle['scenario']
    if scenario == 'raw_no_scaler' or scenario == 'without_cycle':
        return inputs.loc[:, bundle['base_columns']].copy()
    if bundle.get('feature_coordinates') == 'provided_v1':
        coordinates = inputs
    elif 'feature_coordinates' not in bundle and 'normal_scaler' in bundle:
        coordinates = pd.DataFrame(bundle['normal_scaler'].transform(inputs), index=inputs.index, columns=inputs.columns)
    else:
        raise ValueError('Unknown RF feature coordinates; retrain the model')
    if scenario == 'group_pca':
        result = pd.DataFrame(index=inputs.index)
        for group, item in bundle['pcas'].items():
            values = coordinates[item['columns']].to_numpy()
            scores = item['pca'].transform(values)
            for j in range(scores.shape[1]):
                result[f'{group}_PC{j + 1}'] = scores[:, j]
            if len(item['columns']) > scores.shape[1]:
                reconstruction = item['pca'].inverse_transform(scores)
                result[f'{group}_재구성RMSE'] = np.sqrt(np.mean((values - reconstruction) ** 2, axis=1))
        return result
    return pd.concat([inputs.loc[:, bundle['base_columns']], derived_frame(coordinates, bundle['specs'])], axis=1)

def fit_features(Xtrain: pd.DataFrame, ytrain: np.ndarray, scenario: str):
    assert scenario in SCENARIOS and set(ytrain) == {0, 1}
    normal = Xtrain.iloc[np.flatnonzero(ytrain == 0)]
    active = [column for column in Xtrain if normal[column].nunique() > 1]
    base = [column for column in Xtrain if scenario != 'without_cycle' or column not in GROUPS['형체와 전체 주기']]
    bundle = dict(scenario=scenario, input_columns=list(Xtrain.columns), base_columns=base, specs=[], skipped_specs=[], pcas={}, feature_coordinates='provided_v1')
    for spec in DOMAIN_SPEC:
        if spec['group'] not in SCENARIO_GROUPS[scenario]:
            continue
        cols = [column for column in spec['columns'] if column in active]
        if len(cols) < 2:
            bundle['skipped_specs'].append(dict(feature=spec['feature'], reason='fewer than 2 active sensors'))
        else:
            bundle['specs'].append({**spec, 'active_columns': cols})
    if scenario == 'group_pca':
        for group, columns in GROUPS.items():
            cols = [column for column in columns if column in active]
            if cols:
                bundle['pcas'][group] = dict(columns=cols, pca=PCA(n_components=min(2, len(cols)), svd_solver='full').fit(normal[cols].to_numpy()))
    frame = feature_frame(bundle, Xtrain)
    bundle['variance'] = VarianceThreshold(0.0).fit(frame)
    bundle['feature_names'] = list(frame.columns[bundle['variance'].get_support()])
    values = bundle['variance'].transform(frame)
    assert values.shape[1] > 0 and np.isfinite(values).all()
    return (bundle, values)

def transform_features(bundle: dict, inputs: pd.DataFrame) -> np.ndarray:
    values = bundle['variance'].transform(feature_frame(bundle, inputs))
    assert values.shape[1] == len(bundle['feature_names']) and np.isfinite(values).all()
    return values

def make_forest(params: dict) -> RandomForestClassifier:
    return RandomForestClassifier(criterion='gini', bootstrap=True, max_samples=None, min_samples_split=2, ccp_alpha=0.0, oob_score=False, warm_start=False, random_state=SEED, n_jobs=1, **params)

def risk_probability(forest: RandomForestClassifier, values: np.ndarray) -> np.ndarray:
    risk_col = np.flatnonzero(forest.classes_ == 1)
    assert len(risk_col) == 1
    p = forest.predict_proba(values)[:, risk_col[0]]
    assert p.shape == (len(values),) and np.isfinite(p).all()
    assert ((p >= 0) & (p <= 1)).all()
    return p
''', '053.random_forest_feature_library', 'exec'), rf_library)

oc_library = dict(globals())
exec(compile(r'''
GROUPS = {'충전과 전환': ['Injection_Time', 'Filling_Time', 'Cushion_Position', 'Max_Injection_Speed', 'Max_Injection_Pressure', 'Max_Switch_Over_Pressure'], '계량과 가소화': ['Plasticizing_Time', 'Plasticizing_Position', 'Max_Screw_RPM', 'Average_Screw_RPM', 'Max_Back_Pressure', 'Average_Back_Pressure'], '실린더와 호퍼 온도': [f'Barrel_Temperature_{i}' for i in range(1, 7)] + ['Hopper_Temperature'], '금형 온도': ['Mold_Temperature_3', 'Mold_Temperature_4'], '형체와 전체 주기': ['Clamp_Close_Time', 'Clamp_Open_Position', 'Cycle_Time']}
SCENARIOS = {'raw': (), 'legacy_without_cycle': (), 'pressure': ('pressure',), 'plasticizing': ('plasticizing',), 'thermal': ('thermal',), 'domain_all': ('pressure', 'plasticizing', 'thermal'), 'group_pca': ()}
DOMAIN_SPEC = [dict(feature='사출_전환압력_상대편차', group='pressure', operation='difference', columns=['Max_Injection_Pressure', 'Max_Switch_Over_Pressure'], hypothesis='최대 사출압력과 전환압력이 정상 대비 함께 변하는지 확인', limitation='서로 다른 시점의 요약값이며 실제 압력 강하가 아님', source='ARBURG 64/2017 p27'), dict(feature='스크루RPM_상대편차', group='plasticizing', operation='difference', columns=['Max_Screw_RPM', 'Average_Screw_RPM'], hypothesis='가소화 회전수의 최대·평균 상대 변화 불일치', limitation='실제 RPM 폭이나 시간적 변동성으로 해석할 수 없음', source='Sensors 2022 22(13):4792; 제공 변수명'), dict(feature='배압_상대편차', group='plasticizing', operation='difference', columns=['Max_Back_Pressure', 'Average_Back_Pressure'], hypothesis='가소화 배압의 최대·평균 상대 변화 불일치', limitation='실제 배압 차이 또는 불량 인과 효과가 아님', source='Sensors 2022 22(13):4792; 제공 변수명'), dict(feature='배럴_상대수준평균', group='thermal', operation='mean', columns=[f'Barrel_Temperature_{i}' for i in range(1, 7)], hypothesis='활성 배럴 센서의 정상 대비 공통 온도 이동', limitation='평균 섭씨 온도 아님; 센서 위치·목표 온도는 미확인', source='제공 변수명; 공정 가설'), dict(feature='배럴_상대편차표준편차', group='thermal', operation='std', columns=[f'Barrel_Temperature_{i}' for i in range(1, 7)], hypothesis='활성 배럴 센서의 정상 대비 변화가 서로 다른 정도', limitation='물리적 온도 균일도·공간 구배 아님', source='제공 변수명; 공정 가설'), dict(feature='금형_상대수준평균', group='thermal', operation='mean', columns=['Mold_Temperature_3', 'Mold_Temperature_4'], hypothesis='금형 센서의 정상 대비 공통 온도 이동', limitation='센서가 상·하형인지 미확인; 평균 섭씨 온도 아님', source='제공 변수명; 공정 가설'), dict(feature='금형온도_상대편차', group='thermal', operation='difference', columns=['Mold_Temperature_3', 'Mold_Temperature_4'], hypothesis='금형 두 센서의 정상 대비 상대 변화 불일치', limitation='실제 온도차·냉각 효율 아님', source='제공 변수명; 공정 가설')]

def derived_frame(z, specs):
    result = pd.DataFrame(index=z.index)
    for spec in specs:
        values = z[spec['active_columns']]
        operation = spec['operation']
        if operation == 'difference':
            result[spec['feature']] = values.iloc[:, 0] - values.iloc[:, 1]
        elif operation == 'mean':
            result[spec['feature']] = values.mean(axis=1)
        elif operation == 'std':
            result[spec['feature']] = values.std(axis=1, ddof=0)
        else:
            raise ValueError(operation)
    return result

def unscaled_frame(bundle, inputs):
    inputs = inputs[bundle['input_columns']]
    z = pd.DataFrame(bundle['normal_scaler'].transform(inputs), index=inputs.index, columns=inputs.columns)
    if bundle['scenario'] == 'group_pca':
        frame = pd.DataFrame(index=inputs.index)
        for group, item in bundle['pcas'].items():
            values = z[item['columns']].to_numpy()
            scores = item['pca'].transform(values)
            for j in range(scores.shape[1]):
                frame[f'{group}_PC{j + 1}'] = scores[:, j]
            if len(item['columns']) > scores.shape[1]:
                frame[f'{group}_재구성RMSE'] = np.sqrt(np.mean((values - item['pca'].inverse_transform(scores)) ** 2, axis=1))
        return frame
    return pd.concat([z[bundle['base_columns']], derived_frame(z, bundle['specs'])], axis=1)

def fit_features(normal, scenario):
    if scenario not in SCENARIOS:
        raise ValueError(scenario)
    active = [c for c in normal if normal[c].nunique() > 1]
    scaler = StandardScaler().fit(normal)
    bundle = dict(scenario=scenario, input_columns=list(normal), normal_scaler=scaler, specs=[], skipped_specs=[], pcas={})
    for spec in DOMAIN_SPEC:
        if spec['group'] not in SCENARIOS[scenario]:
            continue
        cols = [c for c in spec['columns'] if c in active]
        if len(cols) < 2:
            bundle['skipped_specs'].append({'feature': spec['feature'], 'reason': '활성 센서 2개 미만'})
        else:
            bundle['specs'].append({**spec, 'active_columns': cols})
    bundle['base_columns'] = [c for c in active if scenario != 'legacy_without_cycle' or c not in GROUPS['형체와 전체 주기']]
    if scenario == 'group_pca':
        z = pd.DataFrame(scaler.transform(normal), columns=normal.columns, index=normal.index)
        for group, cols in GROUPS.items():
            cols = [c for c in cols if c in active]
            if cols:
                bundle['pcas'][group] = {'columns': cols, 'pca': PCA(n_components=min(2, len(cols)), svd_solver='full').fit(z[cols].to_numpy())}
    frame = unscaled_frame(bundle, normal)
    bundle['variance'] = VarianceThreshold(0).fit(frame)
    values = bundle['variance'].transform(frame)
    bundle['feature_scaler'] = StandardScaler().fit(values)
    bundle['feature_names'] = list(frame.columns[bundle['variance'].get_support()])
    return (bundle, transform_features(bundle, normal))

def transform_features(bundle, inputs):
    frame = unscaled_frame(bundle, inputs)
    values = bundle['feature_scaler'].transform(bundle['variance'].transform(frame))
    assert np.isfinite(values).all()
    return values

def fit_model(values, nu, multiplier):
    model = OneClassSVM(kernel='rbf', nu=float(nu), gamma=float(multiplier) / values.shape[1], tol=0.0001, max_iter=100000).fit(values)
    assert model.fit_status_ == 0, 'OCSVM 수렴 실패'
    return model
''', '052.ocsvm_feature_library', 'exec'), oc_library)

## 7.3 선택 모델 위치
RF는 053 전체 탐색 산출물을 지정합니다. 모델 생성 환경과 동일한 scikit-learn 버전을 사용하세요.

In [ ]:
# RF는 053 노트북의 전체 탐색 산출물만 사용합니다.
# 다른 위치에 저장했다면 RF_RUNS만 실제 경로로 바꾸세요.
RF_RUNS = {ds: ROOT/f'output/random_forest_{ds}/rf_parameter_search_v1' for ds in ['cn7','rg3']}
# 현재 전체 탐색을 실행한 위치입니다. 저장소 경로에 결과가 있으면 그 경로를 우선 사용합니다.
RF_EXTERNAL_ROOT=Path(r'C:\Users\Playdata\Documents\Codex\2026-10-08\go\work\rf053_results')
for ds in RF_RUNS:
    if not (RF_RUNS[ds]/'model.joblib').exists() and (RF_EXTERNAL_ROOT/ds/'model.joblib').exists():
        RF_RUNS[ds]=RF_EXTERNAL_ROOT/ds
LR_RUNS = {
    'cn7': ROOT/'output/logistic/cn7/lr-20261003T040927Z-66e8d685',
    'rg3': ROOT/'output/logistic/rg3/lr-20261003T040950Z-8c8b16ec',
}
OCSVM_RUNS = {
    'cn7':ROOT/'runtime/cn7/models/ocsvm-6bb288e7f3ba464e',
    'rg3':ROOT/'runtime/rg3/models/ocsvm-657078ff64fd4278',
}
IF_RUNS = {
    'cn7':ROOT/'runtime/cn7/models/if-a7a2fcba54e54d58',
    'rg3':ROOT/'runtime/rg3/models/if-7e69cab8fdd04a17',
}
MODEL_FOLDERS = {'rf':RF_RUNS,'lr':LR_RUNS,'ocsvm':OCSVM_RUNS,'if':IF_RUNS}
SAVED_PREDICTIONS = {
    kind:{ds:Path(paths[ds])/'test_predictions.csv' for ds in ['cn7','rg3']}
    for kind,paths in {'rf':RF_RUNS,'lr':LR_RUNS}.items()
}
SAVED_PREDICTIONS['ocsvm']={ds:ROOT/f'output/report_visualizations/20261006_v4/{ds}_ocsvm_frozen_test.csv' for ds in ['cn7','rg3']}
def selected_info(dataset,kind):
    folder=Path(MODEL_FOLDERS[kind][dataset]);path=folder/'model.joblib'
    if not path.is_file():raise FileNotFoundError(f'{kind.upper()} 산출물 없음: {path}. MODEL_FOLDERS를 확인하세요.')
    sp=folder/'selection_manifest.json';metadata=read_json(sp if sp.exists() else folder/'manifest.json')
    if metadata.get('artifact_sha256') and digest(path)!=metadata['artifact_sha256']:raise ValueError('저장 모델 해시 불일치')
    if kind=='rf' and (metadata.get('setting_count')!=2268 or metadata.get('test_used_for_selection') is not False):raise ValueError('053 전체 탐색 선택 기록이 아닙니다.')
    return dict(dataset=dataset,kind=kind,folder=str(folder),model_sha256=digest(path),metadata=metadata)
def load_selected(dataset,kind):
    folder=Path(MODEL_FOLDERS[kind][dataset]);path=folder/'model.joblib'
    if not path.is_file():raise FileNotFoundError(f'{kind.upper()} 산출물 없음: {path}. 위 MODEL_FOLDERS의 경로를 확인하세요.')
    selection_path=folder/'selection_manifest.json';manifest_path=folder/'manifest.json'
    metadata=read_json(selection_path) if selection_path.exists() else read_json(manifest_path)
    expected=metadata.get('artifact_sha256')
    if expected and digest(path)!=expected:raise ValueError('저장 모델 해시 불일치')
    expected_version=metadata.get('sklearn')
    if expected_version and expected_version!=sklearn.__version__:raise RuntimeError(f'{kind}: 생성 환경 sklearn {expected_version}, 현재 {sklearn.__version__}')
    # 이 저장소에서 생성한 모델만 로드합니다.
    with warnings.catch_warnings():
        warnings.filterwarnings('error',category=sklearn.exceptions.InconsistentVersionWarning)
        artifact=joblib.load(path)
    if kind=='rf':
        if not {'bundle','forest','threshold'}<=set(artifact):raise ValueError('053 전체 탐색 RF의 저장 형식이 아닙니다.')
        if metadata.get('test_used_for_selection') is not False:raise ValueError('RF 설정 선택 기록을 확인하세요.')
        if artifact.get('selection_manifest_sha256')!=digest(selection_path):raise ValueError('RF 선택 기록 해시 불일치')
    elif kind=='lr' and artifact.get('selection_sha256')!=digest(selection_path):raise ValueError('LR 선택 기록 해시 불일치')
    card=dict(dataset=dataset,kind=kind,artifact=artifact,folder=str(folder),model_sha256=digest(path),metadata=metadata)
    return card
def predict_scores(card,x):
    x=validate_inputs(x);b=card['artifact'];kind=card['kind']
    if kind=='rf':values=rf_library['transform_features'](b['bundle'],x);model=b['forest'];s=rf_library['risk_probability'](model,values)
    elif kind=='lr':
        values=oc_library['transform_features'](b['domain'],x) if b.get('domain') else x
        model=b['model'];s=model.predict_proba(values)[:,int(np.flatnonzero(model.classes_==1)[0])]
    elif kind=='ocsvm':s=-b['model'].decision_function(oc_library['transform_features'](b['features'],x))
    elif kind=='if':s=-b['model'].score_samples(b['scaler'].transform(b['variance'].transform(x)))
    else:raise ValueError(kind)
    s=np.asarray(s,dtype=float)
    if len(s)!=len(x) or not np.isfinite(s).all():raise ValueError('모델 점수 길이/유한값 오류')
    return s
def threshold(card):
    if card['kind']=='if':return None
    return float(card['artifact']['threshold'])

In [ ]:
DATASET='cn7'
STATE_ROOT=ROOT/'output/batch_operations_runtime'
BATCH_CSV=None
BATCH_ID=None
LABEL_SOURCE=None
COORDINATES_CONFIRMED=False
DRIFT_LIMIT=0.2
MIN_DRIFT_PATTERNS=30
# 분포 감시 기준은 예시 설정입니다. 현장 기준 자료에서 적합성을 검증해야 합니다.
RUN_INITIALISE=False
RUN_BATCH=False
RUN_RETRAIN=False
RUN_INDEPENDENT_EVALUATION=False
RUN_PROMOTE=False
RUN_ROLLBACK=False

## 7.4 배치 추론·분포 감시·재학습·평가·전환 함수
동일 입력 패턴의 라벨은 최대값으로 집계하며 제품별 원본 연결을 보존합니다. 입력 지문은 제공 숫자값에 근거합니다. 신규 배치가 현장 좌표와 정합되는지 별도로 확인하세요.

In [ ]:
def consolidate_products(frame,label_source=None):
    x=validate_inputs(frame).reset_index(drop=True)
    idcol=next((c for c in ['record_id','product_id'] if c in frame),None)
    if idcol is None:raise ValueError('배치에는 실제 record_id 또는 product_id가 필요합니다.')
    if frame[idcol].isna().any() or frame[idcol].astype(str).duplicated().any():raise ValueError('제품 ID 누락/중복')
    products=x.copy();products['record_id']=frame[idcol].astype(str).to_numpy();products['fingerprint']=fingerprints(x)
    if 'PassOrFail' in frame:
        if not label_source:raise ValueError('실제 검사 라벨 출처가 필요합니다.')
        products['label']=labels(frame.PassOrFail)
    groups=products.groupby('fingerprint',sort=False)
    patterns=groups[FEATURES].first();patterns['product_count']=groups.size()
    if 'label' in products:
        patterns['label']=groups.label.max();patterns['defect_count']=groups.label.sum();patterns['normal_count']=groups.size()-groups.label.sum()
    return products,patterns.reset_index()
def artifact_content_hash(card):
    return hashlib.sha256(joblib.hash(card['artifact']).encode()).hexdigest()
def local_state(state_root,dataset):
    p=Path(state_root)/dataset;p.mkdir(parents=True,exist_ok=True);return p
def status(state_root,dataset):
    state=Path(state_root)/dataset
    return dict(dataset=dataset,state=str(state),models=read_json(state/'active.json') if (state/'active.json').exists() else {},
                batches=[p.name for p in (state/'batches').glob('*') if (p/'manifest.json').exists()],
                candidates=[p.name for p in (state/'candidates').glob('*') if (p/'manifest.json').exists()])
def initialise_models(dataset,state_root,reason):
    if not reason.strip():raise ValueError('초기 모델 지정 사유가 필요합니다.')
    state=local_state(state_root,dataset)
    if (state/'active.json').exists():raise ValueError('이미 초기화됨: 후보 평가 후 전환하세요.')
    active={}
    for kind in ['if','lr','ocsvm','rf']:
        card=load_selected(dataset,kind);folder=state/'models'/run_id(kind);folder.mkdir(parents=True)
        joblib.dump(card,folder/'card.joblib')
        write_json(folder/'manifest.json',dict(kind=kind,source=card['folder'],source_model_sha256=card['model_sha256'],artifact_sha256=digest(folder/'card.joblib'),created_at=timestamp(),reason=reason))
        active[kind]=folder.name
    x,y,_,dev,_=load_dataset(dataset)
    x.loc[dev].assign(label=y[dev],fingerprint=fingerprints(x.loc[dev])).to_csv(state/'training_history.csv',index=False)
    write_json(state/'active.json',active);write_json(state/'events.json',[dict(action='initialise',at=timestamp(),reason=reason,active=active)])
    return active
def active_card(dataset,state_root,kind):
    state=local_state(state_root,dataset);version=read_json(state/'active.json')[kind]
    folder=state/'models'/version;manifest=read_json(folder/'manifest.json')
    if digest(folder/'card.joblib')!=manifest['artifact_sha256']:raise ValueError('운영 모델 파일 손상')
    card=joblib.load(folder/'card.joblib')
    if card['dataset']!=dataset or card['kind']!=kind:raise ValueError('모델 데이터/종류 불일치')
    return card,version
def distribution_shift(reference,new):
    result=[]
    for c in FEATURES:
        ref=reference[c].to_numpy();now=new[c].to_numpy()
        edges=np.unique(np.quantile(ref,[.1,.25,.5,.75,.9]))
        bounds=np.r_[-np.inf,edges,np.inf]
        p=np.histogram(ref,bounds)[0]/len(ref);q=np.histogram(now,bounds)[0]/len(now)
        result.append(dict(feature=c,total_variation=float(.5*np.abs(p-q).sum()),outside_reference=float(((now<ref.min())|(now>ref.max())).mean())))
    return pd.DataFrame(result)
def process_batch(dataset,state_root,frame,batch_id,coordinates_confirmed,label_source=None,drift_limit=.2,min_patterns=30):
    if not coordinates_confirmed:raise ValueError('제공 좌표와 배치 입력 좌표의 정합 확인이 필요합니다.')
    if not isinstance(batch_id,str) or not batch_id or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in batch_id):raise ValueError('배치 ID 형식 오류')
    if not 0<=drift_limit<=1 or min_patterns<1:raise ValueError('분포 감시 조건 오류')
    state=local_state(state_root,dataset);folder=state/'batches'/batch_id
    products,patterns=consolidate_products(frame,label_source)
    content_hash=hashlib.sha256(pd.util.hash_pandas_object(products,index=False).values.tobytes()).hexdigest()
    if folder.exists():
        if not (folder/'manifest.json').exists():raise ValueError('미완료 배치 폴더: 저장 내용을 확인하세요.')
        old=read_json(folder/'manifest.json')
        if old['content_hash']!=content_hash or old['label_source']!=label_source:raise ValueError('같은 배치 ID의 입력/라벨 변경')
        for name,h in old['files'].items():
            if digest(folder/name)!=h:raise ValueError('저장 배치 파일 손상: '+name)
        return old
    # 중복 제품을 다른 배치 ID로 다시 입력하는 것도 거부합니다.
    for past in (state/'batches').glob('*/products.csv'):
        if set(pd.read_csv(past,dtype={'record_id':str}).record_id)&set(products.record_id):raise ValueError('이전에 수집한 제품 ID 중복')
    scores=patterns[['fingerprint','product_count']].copy();versions={}
    for kind in ['if','lr','ocsvm','rf']:
        card,version=active_card(dataset,state_root,kind);versions[kind]=version;s=predict_scores(card,patterns[FEATURES]);scores[kind+'_score']=s
        if kind!='if':scores[kind+'_prediction']=(s>threshold(card)).astype(int)
    scores=scores.sort_values(['if_score','fingerprint'],ascending=[False,True]);scores['if_rank']=np.arange(1,len(scores)+1)
    scored_products=products.merge(scores.drop(columns='product_count'),on='fingerprint',validate='many_to_one').sort_values(['if_rank','record_id'])
    x,y,_,dev,_=load_dataset(dataset);reference=x.loc[dev][y[dev]==0]
    shift=distribution_shift(reference,patterns[FEATURES]);state_value='waiting' if len(patterns)<min_patterns else ('review' if shift.total_variation.max()>drift_limit else 'normal')
    folder.mkdir(parents=True)
    products.to_csv(folder/'products.csv',index=False);patterns.to_csv(folder/'patterns.csv',index=False)
    scores.to_csv(folder/'pattern_predictions.csv',index=False);scored_products.to_csv(folder/'product_predictions.csv',index=False);shift.to_csv(folder/'distribution_shift.csv',index=False)
    manifest=dict(dataset=dataset,batch_id=batch_id,created_at=timestamp(),content_hash=content_hash,label_source=label_source,coordinates_confirmed=True,
                  products=len(products),patterns=len(patterns),versions=versions,status=state_value,drift_limit=drift_limit,min_patterns=min_patterns,
                  ordering='IF 점수 내림차순, 동점은 입력 지문 오름차순',models_applied_to='전체 입력 패턴',combined_filtering_verified=False,
                  files={p.name:digest(p) for p in folder.glob('*.csv')})
    write_json(folder/'manifest.json',manifest);return manifest
def refit_selected(card,x,y):
    x=validate_inputs(x);y=labels(y);new=copy.deepcopy(card);b=new['artifact'];kind=card['kind']
    if kind in ['lr','rf'] and set(y)!={0,1}:raise ValueError('지도학습에는 두 클래스가 필요합니다.')
    if not (y==0).any():raise ValueError('정상 학습 패턴이 없습니다.')
    if kind=='rf':
        bundle,values=rf_library['fit_features'](x,y,b['bundle']['scenario']);b['forest']=clone(b['forest']).fit(values,y);b['bundle']=bundle
    elif kind=='lr':
        if b.get('domain'):b['domain'],_=oc_library['fit_features'](x.loc[y==0],b['scenario']);values=oc_library['transform_features'](b['domain'],x)
        else:values=x
        b['model']=clone(b['model']).fit(values,y)
    elif kind=='ocsvm':
        b['features'],values=oc_library['fit_features'](x.loc[y==0],b['choice']['scenario']);b['model']=oc_library['fit_model'](values,b['choice']['nu'],b['choice']['gamma_multiplier'])
    elif kind=='if':
        normal=x.loc[y==0];b['variance']=VarianceThreshold().fit(normal);values=b['variance'].transform(normal);b['scaler']=StandardScaler().fit(values);b['model']=clone(b['model']).fit(b['scaler'].transform(values))
    new['parent_model_sha256']=card['model_sha256'];new['model_sha256']=artifact_content_hash(new)
    return new
def create_retraining_candidate(dataset,state_root,kind,batch_ids,cause,label_source_confirmed,min_normal=20,min_risk=2):
    if not cause.strip() or not label_source_confirmed:raise ValueError('원인 검토와 실제 검사 라벨 확인이 필요합니다.')
    if not batch_ids or len(set(batch_ids))!=len(batch_ids):raise ValueError('재학습 배치 목록 오류')
    state=local_state(state_root,dataset);parts=[pd.read_csv(state/'training_history.csv',float_precision='round_trip')]
    has_review=False
    for bid in batch_ids:
        folder=state/'batches'/bid;m=read_json(folder/'manifest.json');has_review|=m['status']=='review'
        if not m['label_source']:raise ValueError('비라벨 배치는 재학습에 사용하지 않습니다.')
        if digest(folder/'patterns.csv')!=m['files']['patterns.csv']:raise ValueError('배치 학습 파일 손상')
        p=pd.read_csv(folder/'patterns.csv',float_precision='round_trip')
        if 'label' not in p:raise ValueError('실제 라벨 없음')
        parts.append(p)
    if not has_review:raise ValueError('분포 검토 대상 배치가 없습니다.')
    corpus=pd.concat(parts,ignore_index=True)[FEATURES+['label']]
    corpus['label']=labels(corpus.label);corpus['fingerprint']=fingerprints(corpus[FEATURES])
    corpus=corpus.groupby('fingerprint',sort=False).agg({**{c:'first' for c in FEATURES},'label':'max'}).reset_index()
    x,y,_,dev,test=load_dataset(dataset);protected=set(fingerprints(x.loc[test]))
    for f in (state/'evaluations').glob('*/patterns.csv'):protected.update(pd.read_csv(f).fingerprint)
    if set(corpus.fingerprint)&protected:raise ValueError('학습자료와 평가자료의 입력 패턴 중복')
    n0=int(corpus.label.eq(0).sum());n1=int(corpus.label.eq(1).sum())
    if n0<min_normal or (kind in ['lr','rf'] and n1<min_risk):raise ValueError('재학습 정상/위험 패턴 표본 부족')
    old,parent=active_card(dataset,state_root,kind);new=refit_selected(old,corpus[FEATURES],corpus.label)
    cid=run_id(kind);folder=state/'candidates'/cid;folder.mkdir(parents=True)
    joblib.dump(new,folder/'card.joblib');corpus.to_csv(folder/'corpus.csv',index=False)
    write_json(folder/'manifest.json',dict(id=cid,dataset=dataset,kind=kind,parent=parent,created_at=timestamp(),cause=cause,batch_ids=batch_ids,
              artifact_sha256=digest(folder/'card.joblib'),corpus_sha256=digest(folder/'corpus.csv'),settings='선택 설정과 임계값 유지, 변환과 모델 전체 재적합'))
    return cid
def evaluate_candidate(dataset,state_root,candidate_id,frame,label_source,criteria):
    state=local_state(state_root,dataset);folder=state/'candidates'/candidate_id;m=read_json(folder/'manifest.json')
    if digest(folder/'card.joblib')!=m['artifact_sha256'] or digest(folder/'corpus.csv')!=m['corpus_sha256']:raise ValueError('후보 파일 손상')
    _,p=consolidate_products(frame,label_source)
    if 'label' not in p:raise ValueError('독립 평가는 실제 라벨이 필요합니다.')
    training=pd.read_csv(state/'training_history.csv');training_ids=set(training.fingerprint)
    for f in (state/'candidates').glob('*/corpus.csv'):training_ids.update(pd.read_csv(f).fingerprint)
    if set(p.fingerprint)&training_ids:raise ValueError('독립 평가와 학습 패턴 중복')
    x,y,_,dev,test=load_dataset(dataset)
    if set(p.fingerprint)&set(fingerprints(x.loc[test])):raise ValueError('대회 평가자료는 독립 평가로 재사용할 수 없습니다.')
    kind=m['kind'];old,parent=active_card(dataset,state_root,kind)
    if parent!=m['parent']:raise ValueError('후보의 기준 운영 모델이 변경되었습니다.')
    new=joblib.load(folder/'card.joblib');scores={name:predict_scores(card,p[FEATURES]) for name,card in [('current',old),('candidate',new)]};rows={}
    for name,card in [('current',old),('candidate',new)]:
        rows[name]=score_metrics(p.label,scores[name])
        if kind!='if':rows[name].update(classification_metrics(p.label,scores[name]>threshold(card)))
    if set(p.label)!={0,1}:raise ValueError('독립 평가에 정상/위험 두 클래스가 필요합니다.')
    minimum={'min_AP'} if kind=='if' else {'min_recall','min_precision','max_FPR'}
    if not minimum<=set(criteria) or any(not 0<=v<=1 for v in criteria.values()):raise ValueError('평가 기준을 실행 전에 명시하세요.')
    c=rows['candidate'];o=rows['current']
    passed=(c['AP']>=criteria['min_AP'] and c['AP']>=o['AP']) if kind=='if' else (c['recall']>=criteria['min_recall'] and (c['precision'] or 0)>=criteria['min_precision'] and c['FPR']<=criteria['max_FPR'] and c['F1']>=o['F1'])
    eid=run_id('eval');ev=state/'evaluations'/eid;ev.mkdir(parents=True);p.to_csv(ev/'patterns.csv',index=False)
    p[['fingerprint','label']].assign(current_score=scores['current'],candidate_score=scores['candidate']).to_csv(ev/'predictions.csv',index=False)
    report=dict(id=eid,candidate_id=candidate_id,parent=parent,kind=kind,status='independent',label_source=label_source,created_at=timestamp(),
                criteria=criteria,metrics=rows,passed=bool(passed),candidate_sha256=m['artifact_sha256'],files={p.name:digest(p) for p in ev.glob('*.csv')})
    write_json(ev/'assessment.json',report);return report
def promote_candidate(dataset,state_root,evaluation_id,reason):
    if not reason.strip():raise ValueError('모델 전환 사유가 필요합니다.')
    state=local_state(state_root,dataset);a=read_json(state/'evaluations'/evaluation_id/'assessment.json')
    if not a['passed'] or a['status']!='independent':raise ValueError('독립 평가 기준 미충족')
    for name,h in a['files'].items():
        if digest(state/'evaluations'/evaluation_id/name)!=h:raise ValueError('평가 근거 파일 손상')
    folder=state/'candidates'/a['candidate_id'];m=read_json(folder/'manifest.json')
    if digest(folder/'card.joblib')!=a['candidate_sha256'] or digest(folder/'corpus.csv')!=m['corpus_sha256']:raise ValueError('평가 후 후보 파일 변경')
    active=read_json(state/'active.json')
    if active[a['kind']]!=a['parent']:raise ValueError('평가 후 운영 모델 변경')
    import shutil
    dest=state/'models'/a['candidate_id']
    if dest.exists():raise ValueError('이미 전환된 후보입니다.')
    shutil.copytree(folder,dest)
    # 모든 검증을 마친 뒤 포인터를 한 번에 저장합니다. 이전 모델 파일은 보존합니다.
    active[a['kind']]=a['candidate_id'];write_json(state/'active.json',active)
    pd.read_csv(folder/'corpus.csv',float_precision='round_trip').to_csv(state/'training_history.csv',index=False)
    events=read_json(state/'events.json');events.append(dict(action='promote',kind=a['kind'],old=a['parent'],new=a['candidate_id'],evaluation_id=evaluation_id,reason=reason,at=timestamp()));write_json(state/'events.json',events)
    return active
def rollback_model(dataset,state_root,kind,reason):
    if not reason.strip():raise ValueError('복구 사유가 필요합니다.')
    state=local_state(state_root,dataset);events=read_json(state/'events.json');active=read_json(state/'active.json')
    record=next((e for e in reversed(events) if e['action']=='promote' and e['kind']==kind and e['new']==active[kind]),None)
    if record is None:raise ValueError('현재 모델에 대해 복구할 이전 버전이 없습니다.')
    old=state/'models'/record['old'];m=read_json(old/'manifest.json')
    if digest(old/'card.joblib')!=m['artifact_sha256']:raise ValueError('복구 모델 손상')
    active[kind]=record['old'];write_json(state/'active.json',active)
    events.append(dict(action='rollback',kind=kind,old=record['new'],new=record['old'],reason=reason,at=timestamp()));write_json(state/'events.json',events)
    # 위험 이력은 복구 시에도 지우지 않습니다.
    return active

## 7.5 상태 조회와 초기 모델 등록

In [ ]:
display(status(STATE_ROOT,DATASET))
if RUN_INITIALISE:
    display(initialise_models(DATASET,STATE_ROOT,reason='선택 모델의 로컬 배치 추론 확인'))

## 7.6 신규 배치 추론 및 IF 우선순위
`BATCH_CSV`에는 입력 24개 변수와 실제 record_id/product_id가 필요합니다. 실제 검사 정답이 있으면 PassOrFail과 LABEL_SOURCE를 함께 지정합니다. 설정 셀의 RUN_BATCH를 켠 뒤 실행하세요.

In [ ]:
if RUN_BATCH:
    if BATCH_CSV is None or BATCH_ID is None:raise ValueError('BATCH_CSV와 BATCH_ID를 지정하세요.')
    batch=pd.read_csv(BATCH_CSV,float_precision='round_trip',dtype={'record_id':str,'product_id':str})
    result=process_batch(DATASET,STATE_ROOT,batch,BATCH_ID,COORDINATES_CONFIRMED,LABEL_SOURCE,DRIFT_LIMIT,MIN_DRIFT_PATTERNS)
    display(result)
    display(pd.read_csv(STATE_ROOT/DATASET/'batches'/BATCH_ID/'product_predictions.csv').head(20))

## 7.7 원인 확인 후 재학습 후보 생성
분포 검토 대상 중 실제 검사 라벨이 있는 배치를 지정합니다. 위험 이력은 누적하여 보존하고 평가 입력과 겹치는 패턴은 학습을 거부합니다. 이 단계에서 설정 탐색이나 평가자료에 의한 임계값 조정은 하지 않습니다.

In [ ]:
RETRAIN_KIND='rf'
RETRAIN_BATCH_IDS=[]
RETRAIN_CAUSE=''
LABEL_SOURCE_CONFIRMED=False
CANDIDATE_ID=None
if RUN_RETRAIN:
    CANDIDATE_ID=create_retraining_candidate(DATASET,STATE_ROOT,RETRAIN_KIND,RETRAIN_BATCH_IDS,RETRAIN_CAUSE,LABEL_SOURCE_CONFIRMED)
    print('후보 버전:',CANDIDATE_ID)

## 7.8 독립 평가에서 운영 모델과 후보 비교
신규 실제 검사자료를 사용합니다. 학습 입력 및 이미 관찰한 대회 평가자료와 겹치면 중단합니다. 기준은 실행 전에 지정하며 후보의 평가값을 보고 설정하지 않습니다. IF 후보에는 min_AP 기준을 사용합니다.

In [ ]:
INDEPENDENT_EVALUATION_CSV=None
INDEPENDENT_LABEL_SOURCE=None
EVALUATION_CRITERIA={}  # 분류: min_recall, min_precision, max_FPR / IF: min_AP
ASSESSMENT_ID=None
if RUN_INDEPENDENT_EVALUATION:
    if CANDIDATE_ID is None or INDEPENDENT_EVALUATION_CSV is None:raise ValueError('후보 ID와 독립 평가 파일을 지정하세요.')
    frame=pd.read_csv(INDEPENDENT_EVALUATION_CSV,float_precision='round_trip',dtype={'record_id':str,'product_id':str})
    assessment=evaluate_candidate(DATASET,STATE_ROOT,CANDIDATE_ID,frame,INDEPENDENT_LABEL_SOURCE,EVALUATION_CRITERIA)
    ASSESSMENT_ID=assessment['id'];display(assessment)

## 7.9 검증 후 전환 또는 이전 모델 복구
통과한 독립 평가의 ID와 전환 사유를 지정합니다. 운영 모델은 자동 교체되지 않습니다. 복구 시에도 수집된 위험 이력을 삭제하지 않습니다.

In [ ]:
PROMOTION_REASON=''
ROLLBACK_KIND='rf'
ROLLBACK_REASON=''
if RUN_PROMOTE:
    if ASSESSMENT_ID is None:raise ValueError('독립 평가 ID를 지정하세요.')
    display(promote_candidate(DATASET,STATE_ROOT,ASSESSMENT_ID,PROMOTION_REASON))
if RUN_ROLLBACK:
    display(rollback_model(DATASET,STATE_ROOT,ROLLBACK_KIND,ROLLBACK_REASON))

## 저장 결과와 범위
배치별 원본 제품, 고유 패턴, IF 순위, 모델별 점수·판정과 분포 차이를 저장합니다. 재학습 후보는 학습 입력·설정·부모 버전과 함께 보존하고 독립 평가 결과 및 전환·복구 이력을 기록합니다. 위험 이력 판정은 개별 제품 불량 확정과 다릅니다. 이 노트북에는 실제 생산 권한, 동시 실행 잠금, 생산 시각별 창 관리, 자동 재개 및 제품 검사 성과에 따른 현장 승인 시스템을 추가하지 않았습니다.